### Sampling

Sobol sequences are used for quasi-random sampling in high-dimensional spaces. In this case, suppose we have a tensor of logits of shape `(1, seq_len, N)`, where `N` is the vocab size.

We'll treat this as a `N`-dimensional hypercube problem, and generate `M` Sobol samples in `N` dimensions. Each sample will be a vector of length `N`, with values in the range `[0, 1]`. For each respective sequence position, a Sobol value of `0.0` greatly favoring the highest logit, while a value of `1.0` greatly favoring the lowest logit, and values in between interpolating accordingly.

In [3]:
from torch import Tensor
from torch.nn import functional as F
import torch
import random

In [4]:
seq_len = 100
vocab_size = 64
num_samples = 3

In [5]:
# we assign some indices as the "ground truth" for testing
data = [random.randint(0, vocab_size - 1) for _ in range(seq_len)]
ground_truth = torch.tensor(data).unsqueeze(0)  # (1, N)
ground_truth = F.one_hot(ground_truth, num_classes=vocab_size).squeeze(0)  # (N, vocab_size)

# next we apply some noise to mimic a partially trained model
logits = ground_truth.float() + torch.randn((seq_len, vocab_size)) * 0.7  # (N, vocab_size)
logits = logits.unsqueeze(0)  # (1, N, vocab_size)

In [6]:
# normal softmax sample with temperature as a baseline
def softmax_sample(logits: Tensor, temperature: float, num_samples: int) -> Tensor:
    probs = torch.softmax(logits / temperature, dim=-1)  # (1, N, vocab_size)
    samples = torch.multinomial(probs.squeeze(0), num_samples=num_samples, replacement=True)  # (num_samples, N)
    return samples.transpose(0, 1)  # (N, num_samples)

In [7]:
baseline = softmax_sample(logits, temperature=0.01, num_samples=num_samples)
high_temp = softmax_sample(logits, temperature=1.1, num_samples=num_samples)

In [8]:
def evaluate_sampling_quality(samples, logits):
    """
    Evaluates the diversity and quality of sampled sequences.
    
    Args:
        samples: Tensor of shape (N, seq_len) - The sampled token IDs
        logits:  Tensor of shape (1, seq_len, classes) - The original model outputs
    """
    N, L = samples.shape
    
    # 1. Redundancy: Percentage of Unique Sequences
    # If this is low, you are wasting GPU cycles on identical paths.
    unique_samples = torch.unique(samples, dim=0)
    num_unique = unique_samples.shape[0]
    pct_unique = (num_unique / N) * 100

    # 2. Pairwise Hamming Distance (Diversity)
    # Measures how many tokens differ between every possible pair of samples.
    # We use a subset if N is very large to save memory.
    if N > 128:
        subset = samples[torch.randperm(N)[:128]]
    else:
        subset = samples
    
    # (Num_subset, 1, L) vs (1, Num_subset, L) -> (Num_subset, Num_subset, L)
    diffs = subset.unsqueeze(0) != subset.unsqueeze(1)
    hamming_dist = diffs.float().mean(dim=-1) # Mean over seq_len
    # Take mean of upper triangle to get average pairwise distance
    mask = torch.triu(torch.ones_like(hamming_dist), diagonal=1)
    mean_hamming = (hamming_dist * mask).sum() / mask.sum()

    # 3. Batch Token Entropy (Spread)
    # At each position, how 'spread out' are the N samples?
    # High entropy here means Jittered is doing its job.
    token_counts = torch.zeros(L, logits.shape[-1], device=samples.device)
    for i in range(N):
        token_counts[torch.arange(L), samples[i]] += 1
    
    token_probs = token_counts / N
    # Avoid log(0)
    token_entropy = -(token_probs * torch.log(token_probs + 1e-9)).sum(dim=-1).mean()

    # 4. Path Quality (Log-Likelihood)
    # Does diversity come at the cost of picking "garbage" tokens?
    log_probs = F.log_softmax(logits, dim=-1).squeeze(0) # (L, classes)
    # Gather the log-probs of the actual sampled tokens
    sample_log_probs = torch.gather(log_probs, 1, samples.T).T # (N, L)
    mean_path_lp = sample_log_probs.sum(dim=-1).mean() # Average log-prob of a sequence

    # 5. Cumulative Coverage
    # Total log-probability mass of the unique sequences found.
    # Higher is better for search.
    unique_log_probs = torch.gather(log_probs, 1, unique_samples.T).T
    total_coverage = torch.logsumexp(unique_log_probs.sum(dim=-1), dim=0)

    return {
        "uniqueness_pct": pct_unique, # higher is better
        "mean_hamming_dist": mean_hamming.item(), # higher is better
        "batch_entropy": token_entropy.item(), # higher is better
        "mean_log_prob": mean_path_lp.item(), # too low suggests garbage paths
        "total_log_prob_coverage": total_coverage.item() # higher is better
    }

In [18]:
def stratified(logits, num_samples):
    batch_size, seq_len, vocab_size = logits.shape
    probs = torch.softmax(logits, dim=-1) # (1, N, vocab_size)
    cdf = torch.cumsum(probs, dim=-1) # (1, N, vocab_size)


    bucket_width = 1.0 / num_samples
    bucket_starts = torch.linspace(0, 1 - bucket_width, steps=num_samples)

    jitter = torch.rand(num_samples, seq_len) * bucket_width  # (num_samples, N)
    val_grid = bucket_starts.view(num_samples, 1) + jitter

    shuffled = torch.rand(num_samples, seq_len).argsort(dim=0)
    val_grid = torch.gather(val_grid, 0, shuffled)

    return torch.searchsorted(cdf.squeeze(0), val_grid.transpose(0,1), side='right').transpose(0,1)  # (num_samples, N)

In [19]:
s = stratified(logits, num_samples)

In [20]:
evaluate_sampling_quality(baseline, logits), evaluate_sampling_quality(high_temp, logits), evaluate_sampling_quality(s, logits)

({'uniqueness_pct': 100.0,
  'mean_hamming_dist': 0.026666665449738503,
  'batch_entropy': 0.025460567325353622,
  'mean_log_prob': -268.982421875,
  'total_log_prob_coverage': -267.88323974609375},
 {'uniqueness_pct': 100.0,
  'mean_hamming_dist': 0.9666666984558105,
  'batch_entropy': 1.0524026155471802,
  'mean_log_prob': -397.8146057128906,
  'total_log_prob_coverage': -396.2636413574219},
 {'uniqueness_pct': 100.0,
  'mean_hamming_dist': 1.0,
  'batch_entropy': 1.0986123085021973,
  'mean_log_prob': -394.13623046875,
  'total_log_prob_coverage': -389.5284423828125})

## Filtering

After the sampling process, we have set of samples of shape `(M, seq_len)`, where each entry is an index in the range `[0, vocab_size - 1]`. If we take the one-hot encoding of these samples, we get a tensor of shape `(M, seq_len, vocab_size)`.

Passing this into the energy based model will give us a tensor of shape `(M, num_docs)` where `num_docs` is the number of documents that were packed into the `seq_len` of the samples.

Since we have a tensor called `doc_ids` of shape `(1, seq_len)` which indicates which document in the sequence belongs to, we can use this with the energy scores to find the sampled documents with the best (lowest) energy scores.

In [32]:
def select_top_k_branches(
    input_tensor: Tensor, 
    scores: Tensor, 
    doc_ids: Tensor, 
    k: int
) -> Tensor:
    """
    Args:
        input_tensor: (N, total_seq, classes)
        scores: (N, num_docs) - Lower is better
        doc_ids: (1, total_seq)
        k: Number of top branches to select
        
    Returns:
        refined: (K, total_seq, classes)
    """
    N, total_seq, classes = input_tensor.shape
    
    # 1. Get indices of the best K branches for each document.
    # We want smallest scores, so we use largest=False.
    # topk_indices shape: (K, num_docs)
    _, topk_indices = torch.topk(scores, k, dim=0, largest=False)
    
    # 2. Expand doc_ids to match the sequence length.
    # doc_ids is (1, total_seq), containing values 0 to num_docs-1.
    # We use these values to index into topk_indices.
    
    # topk_indices is (K, num_docs). We want to select columns based on doc_ids.
    # Result shape: (K, total_seq)
    # We squeeze doc_ids to (total_seq,) for indexing.
    expanded_indices = topk_indices[:, doc_ids.squeeze(0)]
    
    # 3. Prepare indices for gathering.
    # input_tensor is (N, total_seq, classes).
    # We need to gather along dim 0 (the N dimension).
    # expanded_indices is (K, total_seq). We need to expand it to (K, total_seq, classes).
    gather_indices = expanded_indices.unsqueeze(-1).expand(-1, -1, classes)
    
    # 4. Gather the data.
    # torch.gather requires the input and index to have the same number of dimensions.
    # We gather from input_tensor along dim 0.
    refined = torch.gather(input_tensor, 0, gather_indices)
    
    return refined

In [43]:
doc_ids = torch.tensor([[0,0,0,1,1,2]])
# doc lengths
doc_lengths = [3,2,1]
_, seq_len = doc_ids.shape
num_samples = 4
vocab_size = 2

In [36]:
mock_sampling = torch.randn(num_samples, seq_len, vocab_size)

In [37]:
scores = torch.tensor([
    [0.1, 0.9, 0.5], # Branch 0 scores
    [0.2, 0.8, 0.6], # Branch 1 scores
    [0.8, 0.2, 0.7], # Branch 2 scores
    [0.9, 0.1, 0.1]  # Branch 3 scores
]) # should be of shape (num_samples, num_docs) which in this case is (4, 3)

In [38]:
refined = select_top_k_branches(input_tensor=mock_sampling, scores=scores, doc_ids=doc_ids, k=2)

In [47]:
# we'll construct manually the expected output for verification
doc_0_best = mock_sampling[0][:doc_lengths[0]]  # best for doc 0
doc_0_second = mock_sampling[1][:doc_lengths[0]]  # second best for doc 0
doc_1_best = mock_sampling[3][doc_lengths[0]:doc_lengths[0]+doc_lengths[1]]  # best for doc 1
doc_1_second = mock_sampling[2][doc_lengths[0]:doc_lengths[0]+doc_lengths[1]]  # second best for doc 1
doc_2_best = mock_sampling[3][doc_lengths[0]+doc_lengths[1]:]  # best for doc 2
doc_2_second = mock_sampling[0][doc_lengths[0]+doc_lengths[1]:]  # second best for doc 2

In [50]:
doc_best = torch.cat([doc_0_best, doc_1_best, doc_2_best], dim=0)
doc_second = torch.cat([doc_0_second, doc_1_second, doc_2_second], dim=0)

In [52]:
expected = torch.stack([doc_best, doc_second], dim=0)

In [53]:
refined.shape, expected.shape

(torch.Size([2, 6, 2]), torch.Size([2, 6, 2]))

In [54]:
torch.allclose(refined, expected)

True